### 1. Data overview

- Inspect feature types (categorical, continuous, ordinal, binary)
- Check target distribution (class balance)

## 0. Data Loading

In [11]:
import numpy as np
from helper_functions.A_loading_helpers import load_csv_data
import os
import helper_functions.C_preprocessing_helpers as ph

In [2]:
data_path = 'data'
x_train, x_test, y_train, train_ids, test_ids =  load_csv_data(data_path, sub_sample=False)

x_train_raw = x_train.copy()

In [3]:
x_train.shape


(328135, 321)

# 1. Categorizing Features - continuous or categorical

**Tutorial**:  
- You can create a dictionary using `build_feature_dictionary()` that contains tells you what features in the training data are continuous or categorical and so on.  
- Here is example usage where I retrieve the indices / positions of the features that are categorical.  
- you can tell it if you want to know:  
    - `ordinal`: so discrete scale like 1,2,3,4  
    - `categorical`: like green, red, blue  
    - `continuous`: like values between: 1 and 99  
    - `continuous_but_null_also_a_number`: like continuous but the highest number means "no answer"  
    - `not_displayed_or_unrelated`: where all values are Null or the category is things like "Call is on Landline"  
- And yes I did extract this info by hand from that god-forsaken pdf because I like suffering.


In [14]:
classes_path = 'data/feature_properties/feature_classes.json'
feature_names_path = 'data/feature_properties/feature_names.csv'

feature_classes_dictionary = ph.build_feature_dictionary(classes_path, feature_names_path)
print(feature_classes_dictionary['categorical']['indices'])

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 21, 22, 31, 32, 33, 35, 36, 37, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 51, 52, 54, 55, 57, 58, 59, 62, 65, 66, 67, 68, 69, 70, 71, 72, 73, 75, 88, 89, 92, 96, 97, 98, 99, 101, 103, 104, 105, 107, 109, 110, 117, 118, 119, 120, 123, 124, 125, 126, 127, 128, 129, 131, 133, 134, 135, 136, 137, 142, 143, 145, 147, 156, 157, 158, 159, 160, 161, 162, 165, 166, 167, 168, 170, 171, 173, 175, 177, 178, 180, 182, 183, 185, 186, 187, 188, 190, 191, 192, 194, 195, 197, 199, 200, 201, 202, 203, 204, 206, 215, 216, 217, 218, 219, 224, 225, 226, 228, 232, 233, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 261, 262, 264, 266, 267, 268, 269, 270, 271, 272, 275, 276, 279, 280, 281, 282, 283, 284, 285, 299, 311, 312, 313, 314, 315, 316, 320, 321]


In [15]:
print(feature_classes_dictionary['categorical']['names'])

['_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO', '_PSU', 'CTELENUM', 'PVTRESD1', 'COLGHOUS', 'STATERES', 'CELLFON3', 'LADULT', 'CADULT', 'PVTRESD2', 'HLTHPLN1', 'PERSDOC2', 'MEDCOST', 'BPHIGH4', 'BPMEDS', 'BLOODCHO', 'TOLDHI2', 'CVDSTRK3', 'ASTHMA3', 'ASTHNOW', 'CHCSCNCR', 'CHCOCNCR', 'CHCCOPD1', 'HAVARTH3', 'ADDEPEV2', 'CHCKIDNY', 'DIABETE3', 'SEX', 'MARITAL', 'RENTHOM1', 'NUMHHOL2', 'CPDEMO1', 'VETERAN3', 'EMPLOY1', 'INTERNET', 'PREGNANT', 'QLACTLM2', 'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON', 'SMOKE100', 'STOPSMK2', 'EXERANY2', 'EXRACT11', 'EXRACT21', 'LMTJOIN3', 'ARTHDIS2', 'ARTHSOCL', 'JOINPAIN', 'FLUSHOT6', 'IMFVPLAC', 'PNEUVAC3', 'HIVTST6', 'WHRTST10', 'PREDIAB1', 'INSULIN', 'DIABEYE', 'DIABEDU', 'CAREGIV1', 'CRGVREL1', 'CRGVPRB1', 'CRGVPERS', 'CRGVHOUS', 'CRGVMST2', 'CRGVEXPT', 'VIDFCLT2', 'VIREDIF3', 'VINOCRE2', 'VIINSUR2', 'VICTRCT4', 'VIGLUMA2', 'VIMACDG2', 'CIMEMLOS', 'CDDISCUS', 'WTCHSALT', 'DRADVISE', 'ASATTACK', 'HAREH


- here arrays of the feature names are produced based on data type (continuous, cat)
- as well as the indices (which columns in the numpy xrain array they correspond to)

---

- For ordinal: usually its like 1,2,3,4 and then 7 or 9 for 'dont know' or 'refused'. We replace this value with 'null'. The feature _AGEYR65 doesnt have a nice gap.
- there are variables llike _FRUITEX that tell you wether the fruit responses of that person should be excluded. 

---

## 1.1 cleaning up


### 1.1.1 Ordinal Features

for the class'ordinal' figure out where the 'gap' is and replac the highest value with Null  (1,2,3,9 --> 9 is Null because it corresponds to "no answer"

In [5]:
ordinal_indices = feature_classes_dictionary['ordinal']['indices']
 
for j in ordinal_indices:
    x_train[:, j] = ph.clean_ordinal_feature(x_train[:, j])

### 1.1.2 Continuous variables with null as a number

For the class continuous_but_null_also_a_number the highest number is sometimes "no answer" sometimes something else. Check that and replace with "null"

In [6]:
feature_classes_dictionary = ph.build_feature_dictionary(classes_path, feature_names_path)
print(feature_classes_dictionary['continuous_but_null_also_a_number']['names'])
print(feature_classes_dictionary['continuous_but_null_also_a_number']['indices'])

['HHADULT', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'ALCDAY5', 'AVEDRNK2', 'DRNK3GE5', 'MAXDRNKS', 'FRUITJU1', 'FRUIT1', 'FVBEANS', 'FVGREEN', 'FVORANG', 'VEGETAB1', 'EXEROFT1', 'EXERHMM1', 'EXEROFT2', 'EXERHMM2', 'STRENGTH', 'FLSHTMY2', 'HIVTSTD3', 'BLDSUGAR', 'FEETCHK2', 'DOCTDIAB', 'CHKHEMO3', 'LONGWTCH', 'ASTHMAGE', 'ASERVIST', 'ASDRVIST', 'ASRCHKUP', 'ASACTLIM', 'SCNTWRK1', 'ADPLEASR', 'ADDOWN', 'ADSLEEP', 'ADENERGY', 'ADEAT1', 'ADFAIL', 'ADTHINK', 'ADMOVE', 'DROCDY3_', '_DRNKWEK', 'MAXVO2_', 'FC60_', 'PAFREQ1_', 'PAFREQ2_']
[26, 28, 29, 30, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 90, 91, 93, 94, 95, 102, 106, 111, 112, 113, 114, 144, 146, 148, 149, 150, 151, 196, 207, 208, 209, 210, 211, 212, 213, 214, 263, 265, 288, 289, 294, 295]


In [7]:
# Load feature metadata
classes_path = 'data/feature_properties/feature_classes.json'
feature_names_path = 'data/feature_properties/feature_names.csv'
feature_classes_dictionary = ph.build_feature_dictionary(classes_path, feature_names_path)


# Apply cleaning
from data.feature_properties.cleaning_rules_continuous import cleaning_rules  # the big dictionary we saved earlier
x_train = ph.apply_cleaning_continuous_features(x_train, feature_classes_dictionary, cleaning_rules)

### 1.1.3 Categorical One Hot Encoding

### 1.1.4 Remove not displayyed features

remove_indices = feature_classes_dictionary['not_displayed_or_unrelated']['indices']
X_cleaned = np.delete(X, remove_indices, axis=1)


### 2. Data quality

- Missing values (counts & % per column) --> how will we impute them? Drop, Mean or regressioN????


In [8]:
count_1 = np.sum(y_train == 1)
count_minus1 = np.sum(y_train == -1)

print("Number of 1s:", count_1)
print("Number of -1s:", count_minus1)
print("ratio of one over minus one:", count_1/ count_minus1)

Number of 1s: 28975
Number of -1s: 299160
ratio of one over minus one: 0.09685452600615055


In [9]:
def count_missing(data):
    # Initialize the counter at 0
    missing = 0
    for row in data:
        for item in row:
            if item == '' or item is None or np.isnan(item):   # Check for missing values
                missing += 1
    return missing

# Assuming we have the following data (2D list)

total_entries = len(x_train) * len(x_train[0])   # Calculate total entries (rows * columns)
missing_count = count_missing(x_train)       # Count missing values
percentage_missing = (missing_count / total_entries) * 100  # Percentage calculation

print("Total number of missing values in the cleaned data is {} or {:.2f}%".format(missing_count, percentage_missing))

Total number of missing values in the cleaned data is 48277459 or 45.83%


In [10]:
# Assuming we have the following data (2D list)

total_entries = len(x_train_raw) * len(x_train_raw[0])   # Calculate total entries (rows * columns)
missing_count = count_missing(x_train_raw)       # Count missing values
percentage_missing = (missing_count / total_entries) * 100  # Percentage calculation

print("Total number of missing values in the raw data is {} or {:.2f}%".format(missing_count, percentage_missing))

Total number of missing values in the raw data is 47175779 or 44.79%
